## 🛡️ Constraints en Delta Tables

Cuando diseñamos tablas en una base de datos relacional o en un Data Warehouse, es habitual definir reglas que garanticen la integridad de los datos.

Estas reglas reciben el nombre de **Constraints** y permiten establecer condiciones que la información debe cumplir al momento de ser almacenada.

En Delta Lake también podemos definir constraints, aunque su comportamiento es diferente al de un motor relacional tradicional. Databricks los clasifica en dos grandes categorías: 

* **✅ Enforced Constraints**

Los **Enforced Constraints** son restricciones que **sí son validadas por Delta Lake**. Si un registro incumple alguna de estas reglas, la operación de escritura será rechazada.

Actualmente, los principales constraints de este tipo son:
    
* **✔️ CHECK**: Permite definir una condición lógica que deben cumplir los valores de una columna. Ejemplos:

    * Edad mayor o igual a 18
    * Precio mayor que 0
    * Cantidad positiva
    
    Si la condición no se cumple, el registro no será almacenado.


* **🚫 NOT NULL**: Obliga a que una columna siempre contenga un valor. No es posible insertar registros con valores `NULL` en columnas que posean este constraint.

---

* **ℹ️ Informational Constraints**

Los **Informational Constraints** tienen un propósito diferente.

En lugar de validar los datos durante la escritura, **documentan el modelo de datos** y describen las relaciones que deberían existir entre las tablas.

Actualmente, Databricks no impide que los datos incumplan estas restricciones. Entre ellos encontramos:

* **🔑 PRIMARY KEY (PK)**: Indica cuál debería ser la clave primaria de la tabla. No evita la inserción de registros duplicados.

* **🔗 FOREIGN KEY (FK)**: Describe la relación lógica entre dos tablas. Sin embargo, Databricks no valida que exista la fila correspondiente en la tabla relacionada.

* **⭐ UNIQUE**: Indica que una columna debería contener valores únicos.

Actualmente esta restricción es únicamente informativa y no impide la inserción de valores repetidos.

---

### 🤔 ¿Por qué ocurre esto?

A diferencia de los motores OLTP tradicionales, Databricks está diseñado principalmente para cargas analíticas de gran volumen.

Su prioridad es ofrecer:

* 📈 Alto rendimiento en lectura
* ⚡ Procesamiento distribuido
* 📦 Escalabilidad sobre grandes volúmenes de datos

Por ello, muchas de las validaciones de integridad suelen implementarse previamente dentro de los procesos de ingestión o transformación de datos.

---

### 💡 Buenas prácticas

Cuando sea necesario garantizar una alta calidad de los datos, es recomendable:

* ✅ Validar la información antes de escribirla en la Delta Table.
* ✅ Aplicar reglas de negocio durante los procesos ETL o ELT.
* ✅ Utilizar operaciones como `MERGE` para controlar inserciones, actualizaciones y evitar inconsistencias.

De esta manera, los constraints complementan la calidad de los datos, mientras que la lógica de negocio continúa siendo responsabilidad de los pipelines de Data Engineering.

### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("09Constraints").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

#### 🔍 ¿Qué acaba de ocurrir?

Acabamos de crear una Spark Session. Esta sesión representa nuestro punto de entrada hacia:
* 🏗️ Apache Spark
* 🏗️ Databricks Runtime
* 🏗️ Delta Lake
* 🏗️ Unity Catalog

### ✅ Enforced Constraints


In [0]:
## CONSTRAINT - CHECK

### PASO 1. DEFINIR DELTA TABLE

spark.sql("""
          CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.products_check
          (
              product_id INT,
              description VARCHAR(30),
              stock INT
          )
          USING DELTA;
          """)
print("Delta Table definida correctamente")

### PASO 2. DEFINIR CONSTRAINT

spark.sql("""
          
          ALTER TABLE catalog_databricks_2026_de.schema_databricks_2026_de.products_check
          ADD CONSTRAINT stock_mayor_cero
          CHECK (stock>0)
          """)
print("Constraint definido correctamente")

### PASO 3. VALIDAR CONSTRAINT

spark.sql("""
          INSERT INTO catalog_databricks_2026_de.schema_databricks_2026_de.products_check
          (product_id,description,stock) VALUES(
              1,'LG TV 27"',-25
          )
          """)
print("Saldrá el siguiente error: [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint stock_mayor_cero (stock > 0) violated by row with values:")

In [0]:
## CONSTRAINT - NOT NULL

### PASO 1. DEFINIR DELTA TABLE + CONSTRAINT

spark.sql("""
          CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.products_not_null
          (
              product_id INT NOT NULL,
              description VARCHAR(30) NOT NULL,
              stock INT NOT NULL
          )
          USING DELTA;
          """)
print("Delta Table y Constraints definidos correctamente")

# ### PASO 2. VALIDAR CONSTRAINT

spark.sql("""
          INSERT INTO catalog_databricks_2026_de.schema_databricks_2026_de.products_not_null
          (product_id,description,stock) VALUES(
              1,null,null
          )
          """)
print("Saldrá el siguiente error: [DELTA_NOT_NULL_CONSTRAINT_VIOLATED] NOT NULL constraint violated for column: description.")

### ℹ️ Informational Constraints

In [0]:
## CONSTRAINT - PRIMARY KEY

### PASO 1. DEFINIR DELTA TABLE + CONSTRAINT

spark.sql("""
          CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.products_pk
          (
              product_id INT,
              description VARCHAR(30),
              stock INT,
              categoria CHAR(1),
              CONSTRAINT ProductsPK 
              PRIMARY KEY (product_id)
          )
          USING DELTA;
          """)
print("Delta Table y Constraints definidos correctamente")

### PASO 2. VALIDAR CONSTRAINT

spark.sql("""
          INSERT INTO catalog_databricks_2026_de.schema_databricks_2026_de.products_pk
          (product_id,description,stock,categoria) VALUES
          (1,'Producto A',10,'A')
          """)
print("Se realizó correctamente el registro 1.")

spark.sql("""
          INSERT INTO catalog_databricks_2026_de.schema_databricks_2026_de.products_pk
          (product_id,description,stock,categoria) VALUES
          (1,'Producto B',20,'C')
          """)

print("Se realizó correctamente el registro 2.")

### PASO 3. VERIFICAR DATOS REGISTRADOS
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.products_pk"))
#### Resultado: Verificamos que existen datos duplicados, a pesar de Constraint PK.


In [0]:
## CONSTRAINT - FOREIGN KEY

### PASO 1. DEFINIR DELTA TABLE + CONSTRAINT

spark.sql("""
          CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.categories_fk
          (
              categorie_code CHAR(1),
              description VARCHAR(30),
              CONSTRAINT CategoriePK 
              PRIMARY KEY (categorie_code)
          )
          USING DELTA;
          """)
print("Delta Table y Constraints definidos correctamente")

# ### PASO 2. MODIFICAR TABLA RELACIONADA

spark.sql("""
          ALTER TABLE catalog_databricks_2026_de.schema_databricks_2026_de.products_pk
          ADD CONSTRAINT CategorieFK 
          FOREIGN KEY (categoria) REFERENCES 
          catalog_databricks_2026_de.schema_databricks_2026_de.categories_fk (categorie_code)
          """)
print("Se agregó correctamente el constraint a la Tabla Products_PK.")

### PASO 3. REGISTRAR INFORMACIÓN
spark.sql("""
          INSERT INTO catalog_databricks_2026_de.schema_databricks_2026_de.categories_fk
          (categorie_code,description) VALUES
          ('A','Categoria A'),
          ('B','Categoria B'),
          ('C','Categoria C')
          """)

print("Se realizó correctamente los registros.")


In [0]:
## CONSTRAINT - UNIQUE

### PASO 1. DEFINIR DELTA TABLE + CONSTRAINT

spark.sql("""
          CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.products_unique
          (
              product_id INT,
              description VARCHAR(30) UNIQUE,
              stock INT
          )
          USING DELTA;
          """)
print("Delta Table y Constraints definidos correctamente")

### PASO 2. VALIDAR CONSTRAINT

spark.sql("""
          INSERT INTO catalog_databricks_2026_de.schema_databricks_2026_de.products_unique
          (product_id,description,stock) VALUES
          (1,'Producto A',10)
          """)
print("Se realizó correctamente el registro 1.")

spark.sql("""
          INSERT INTO catalog_databricks_2026_de.schema_databricks_2026_de.products_unique
          (product_id,description,stock) VALUES
          (2,'Producto A',20)
          """)
print("Se realizó correctamente el registro 2.")

### PASO 3. VERIFICAR DATOS REGISTRADOS
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.products_unique"))
#### Resultado: Verificamos que existen datos duplicados, a pesar de Constraint Unique.
